# PyReceipt: Benchmark Incrociato e Heatmap di Accuratezza

Questo notebook esegue il benchmark incrociato tra tutti i **Motori OCR** (Tesseract, RapidOCR, EasyOCR) e i **Metodi di Parsing Avanzati** su 50 campioni del dataset **SROIE2019**.

### Metodi di Parsing a Confronto:
1. **Regex Baseline (1D)**: Parsing lineare a espressioni regolari (standard library).
2. **Spatial 2D Box Parser (Metodo 1)**: Clustering geometrico 2D su coordinate $(X, Y)$ con allineamento delle righe e ray-casting orizzontale (< 30 MB RAM).
3. **LayoutLM Document AI (Metodo 2)**: Modello Transformer Multimodale con 2D Spatial Attention (`impira/layoutlm-document-qa`).

In [1]:
import json
import os
from pathlib import Path
import re
import time
from typing import Dict, List, Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Import OCR Adapters
from pyreceipt.adapters.tesseract_ocr import TesseractOCRAdapter
from pyreceipt.adapters.paddle_ocr import RapidOCRAdapter
from pyreceipt.adapters.easy_ocr import EasyOCRAdapter

# Import Parser Methods
from pyreceipt.core.parser import RegexReceiptParser
from pyreceipt.adapters.spatial_2d_parser import Spatial2DBoxParser
from pyreceipt.adapters.layoutlm_parser import LayoutLMReceiptParser

## 1. Funzioni di Supporto per il Calcolo dell'Accuratezza

In [2]:
def parse_ground_truth(entity_file: Path) -> Dict[str, Any]:
    with open(entity_file, 'r', encoding='utf-8', errors='ignore') as f:
        data = json.load(f)
    raw_tot = str(data.get('total', '0')).replace(',', '').replace('$', '').replace('RM', '').strip()
    try:
        gt_tot = float(raw_tot)
    except Exception:
        gt_tot = 0.0
    return {
        'company': data.get('company', '').strip(),
        'date': data.get('date', '').strip(),
        'total': gt_tot,
    }

def is_total_match(parsed: float, gt: float) -> bool:
    if gt <= 0:
        return True
    return abs(parsed - gt) < 0.01

def is_date_match(parsed: str, gt: str) -> bool:
    if not gt:
        return True
    p_norm = re.sub(r'[^\d]', '', parsed)
    g_norm = re.sub(r'[^\d]', '', gt)
    return p_norm == g_norm if (p_norm and g_norm) else parsed == gt

## 2. Esecuzione del Benchmark Incrociato su 50 Campioni

In [3]:
dataset_dir = Path('/Users/manu/Datasets/SROIE2019/test')
entities_dir = dataset_dir / 'entities'
img_dir = dataset_dir / 'img'

entity_files = sorted(list(entities_dir.glob('*.txt')))[:50]
total_samples = len(entity_files)
print(f'Campioni in valutazione: {total_samples}')

ocr_engines = {
    'Tesseract OCR': TesseractOCRAdapter(lang='eng'),
    'RapidOCR (ONNX)': RapidOCRAdapter(),
    'EasyOCR (CRAFT)': EasyOCRAdapter(lang_list=['en']),
}

# Step 1: Estrazione Bounding Box 2D
ocr_boxes = {name: {} for name in ocr_engines}
ocr_texts = {name: {} for name in ocr_engines}

for ocr_name, ocr_adapter in ocr_engines.items():
    print(f'Estrazione con: {ocr_name}...')
    for ent_file in entity_files:
        sample_id = ent_file.stem
        img_file = img_dir / f'{sample_id}.jpg'
        boxes = ocr_adapter.extract_boxes(str(img_file))
        ocr_boxes[ocr_name][sample_id] = boxes
        ocr_texts[ocr_name][sample_id] = '\n'.join(b['text'] for b in boxes)

# Step 2: Inizializzazione Parser
regex_parser = RegexReceiptParser(lang_code='en')
spatial_parser = Spatial2DBoxParser()
layoutlm_parser = LayoutLMReceiptParser()

parser_cols = ['Regex Baseline (1D)', 'Spatial 2D Box Parser (Metodo 1)', 'LayoutLM Document AI (Metodo 2)']
matrix_total = pd.DataFrame(index=list(ocr_engines.keys()), columns=parser_cols, dtype=float)
matrix_date = pd.DataFrame(index=list(ocr_engines.keys()), columns=parser_cols, dtype=float)

# Step 3: Valutazione Incrociata
for ocr_name in ocr_engines:
    # Regex
    tot_r, date_r = 0, 0
    for ent_file in entity_files:
        sample_id = ent_file.stem
        gt = parse_ground_truth(ent_file)
        rec = regex_parser.parse(ocr_texts[ocr_name][sample_id])
        if is_total_match(rec.total, gt['total']): tot_r += 1
        if is_date_match(rec.date, gt['date']): date_r += 1
    matrix_total.loc[ocr_name, 'Regex Baseline (1D)'] = (tot_r / total_samples) * 100
    matrix_date.loc[ocr_name, 'Regex Baseline (1D)'] = (date_r / total_samples) * 100

    # Metodo 1: Spatial 2D Box
    tot_s, date_s = 0, 0
    for ent_file in entity_files:
        sample_id = ent_file.stem
        gt = parse_ground_truth(ent_file)
        rec = spatial_parser.parse(ocr_boxes[ocr_name][sample_id])
        if is_total_match(rec.total, gt['total']): tot_s += 1
        if is_date_match(rec.date, gt['date']): date_s += 1
    matrix_total.loc[ocr_name, 'Spatial 2D Box Parser (Metodo 1)'] = (tot_s / total_samples) * 100
    matrix_date.loc[ocr_name, 'Spatial 2D Box Parser (Metodo 1)'] = (date_s / total_samples) * 100

# Metodo 2: LayoutLM (Diretto su immagine)
tot_l, date_l = 0, 0
for ent_file in entity_files:
    sample_id = ent_file.stem
    gt = parse_ground_truth(ent_file)
    img_file = img_dir / f'{sample_id}.jpg'
    rec = layoutlm_parser.parse_image(str(img_file))
    if is_total_match(rec.total, gt['total']): tot_l += 1
    if is_date_match(rec.date, gt['date']): date_l += 1

for ocr_name in ocr_engines:
    matrix_total.loc[ocr_name, 'LayoutLM Document AI (Metodo 2)'] = (tot_l / total_samples) * 100
    matrix_date.loc[ocr_name, 'LayoutLM Document AI (Metodo 2)'] = (date_l / total_samples) * 100

print('Valutazione completata!')

## 3. Matrice dei Risultati (%)

In [4]:
print('=== ACCURATEZZA TOTALE PREZZO (%) ===')
display(matrix_total)

print('\n=== ACCURATEZZA DATA (%) ===')
display(matrix_date)

## 4. Visualizzazione Heatmap Comparativa (Seaborn)

In [5]:
plt.figure(figsize=(13, 5))
sns.heatmap(
    matrix_total,
    annot=True,
    fmt='.1f',
    cmap='YlGnBu',
    cbar_kws={'label': 'Accuratezza Totale (%)'},
    linewidths=2,
    annot_kws={'size': 14, 'weight': 'bold'},
    vmin=40,
    vmax=80,
)
plt.title('SROIE2019: Accuratezza Estrazione Totale (%) - OCR x Metodo di Parsing', fontsize=14, weight='bold', pad=14)
plt.ylabel('Motore OCR', fontsize=12, weight='bold')
plt.xlabel('Metodo di Parsing', fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

In [6]:
plt.figure(figsize=(13, 5))
sns.heatmap(
    matrix_date,
    annot=True,
    fmt='.1f',
    cmap='Greens',
    cbar_kws={'label': 'Accuratezza Data (%)'},
    linewidths=2,
    annot_kws={'size': 14, 'weight': 'bold'},
    vmin=40,
    vmax=90,
    )
plt.title('SROIE2019: Accuratezza Estrazione Data (%) - OCR x Metodo di Parsing', fontsize=14, weight='bold', pad=14)
plt.ylabel('Motore OCR', fontsize=12, weight='bold')
plt.xlabel('Metodo di Parsing', fontsize=12, weight='bold')
plt.tight_layout()
plt.show()